In [ ]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
from tkinter import ttk


# Load the tourism dataset
data = pd.read_csv("cleaned_tourist_data.csv")

cities = sorted(data["City"].dropna().astype(str).unique())
types = sorted(data["Type"].dropna().astype(str).unique())
attractions = sorted(data["Name"].dropna().astype(str).unique())

In [ ]:
import matplotlib.pyplot as plt

# Count number of attractions for each type
type_counts = data["Type"].value_counts().head(10)

# Create pie chart
plt.figure(figsize=(8, 8))

plt.pie(
    type_counts,
    labels=type_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Distribution of Top 10 Tourist Attraction Types")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Count attractions in each city
city_counts = data["City"].value_counts().head(15)

plt.figure(figsize=(12, 6))

city_counts.plot(kind="bar")

plt.title("Top 15 Cities by Number of Tourist Attractions")
plt.xlabel("City")
plt.ylabel("Number of Attractions")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Count attractions for each recommended visiting time
time_counts = data["Best Time to visit"].value_counts()

# Create bar chart
plt.figure(figsize=(10, 6))

time_counts.plot(kind="bar")

# Add title and labels
plt.title("Best Time to Visit Tourist Attractions")
plt.xlabel("Best Time to Visit")
plt.ylabel("Number of Attractions")

# Rotate category names for readability
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Calculate average entrance fee for each attraction type
fee_by_type = data.groupby("Type")["Entrance Fee in INR"].mean()

# Select top 10 attraction types by average fee
fee_by_type = fee_by_type.sort_values(ascending=False).head(10)

# Create bar chart
plt.figure(figsize=(12, 6))

fee_by_type.plot(kind="bar")

# Title and labels
plt.title("Top 10 Tourist Attraction Types by Average Entrance Fee")
plt.xlabel("Attraction Type")
plt.ylabel("Average Entrance Fee (INR)")

# Rotate labels
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
#DESIGN + HELPER FUNCTIONS


#COLOURS
BG = "#F4F7F6"
DARK = "#173F3A"
GREEN = "#2E7D6E"
GREEN_DARK = "#236356"
CARD = "#FFFFFF"
TEXT = "#172B4D"
MUTED = "#667085"
BORDER = "#E4E7EC"
LIGHT_GREEN = "#E8F3F0"


#CARD
def card(parent):
    return tk.Frame(parent,bg=CARD,highlightbackground=BORDER,highlightthickness=1)

#BUTTON-
def button(parent, text, command):
    b = tk.Button(parent,text=text,command=command,font=("Arial", 10, "bold"),bg=GREEN,fg="white",activebackground=GREEN_DARK,activeforeground="white",
        relief="flat",cursor="hand2",padx=16,pady=7)
    b.bind("<Enter>",lambda e: b.config(bg=GREEN_DARK))
    b.bind("<Leave>",lambda e: b.config(bg=GREEN))
    return b

#SCROLLABLE PAGE
def scroll_page(parent):
    canvas = tk.Canvas(parent,bg=BG,highlightthickness=0)
    bar = ttk.Scrollbar(parent,orient="vertical",command=canvas.yview)
    content = tk.Frame(canvas,bg=BG)
    window = canvas.create_window((0, 0),window=content,anchor="nw")
    content.bind("<Configure>",lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
    canvas.bind("<Configure>",lambda e: canvas.itemconfig(window,width=e.width))
    canvas.configure(yscrollcommand=bar.set)
    canvas.pack(side="left",fill="both",expand=True)
    bar.pack(side="right",fill="y")

    # Mouse wheel scrolling
    def mouse_scroll(event):
        widget = canvas.winfo_containing(event.x_root,event.y_root)

        if widget is None:
            return

        current = widget
        belongs_to_page = False

        while current is not None:
            if current == content or current == canvas:
                belongs_to_page = True
                break

            try:
                current = current.master
            except AttributeError:
                break

        if belongs_to_page:
            units = -1 * (event.delta // 120)
            if units == 0:
                units = -1 if event.delta > 0 else 1
            canvas.yview_scroll(units,"units")

    canvas.bind_all("<MouseWheel>",mouse_scroll,add="+")

    # Linux scrolling support
    canvas.bind_all("<Button-4>",lambda e: mouse_scroll(
            type("Event",(),{
                    "x_root": e.x_root,
                    "y_root": e.y_root,
                    "delta": 120
                }
            )()
        ),
        add="+"
    )

    canvas.bind_all("<Button-5>",lambda e: mouse_scroll(type("Event",(),{
                    "x_root": e.x_root,"y_root": e.y_root,"delta": -120
                }
            )()
        ),
        add="+"
    )

    return canvas, content


#CLEAR WIDGET
def clear(widget):

    for item in widget.winfo_children():
        item.destroy()

#SCROLL TO WIDGET
def scroll_to(canvas, widget, content):

    canvas.update_idletasks()
    position = widget.winfo_y() / max(1,content.winfo_height())
    canvas.yview_moveto(position)

#ATTRACTION DETAILS
def details_text(a):
    return (
        f"🏛  {a['Name']}\n\n"
        f"📍 City: {a['City']}\n"
        f"State: {a['State']}\n"
        f"Type: {a['Type']}\n\n"
        f"⭐ Google Rating: {a['Google review rating']}\n"
        f"💰 Entrance Fee: ₹{a['Entrance Fee in INR']}\n"
        f"⏱ Time Needed: "
        f"{a['time needed to visit in hrs']} hours\n"
        f"🌤 Best Time to Visit: "
        f"{a['Best Time to visit']}\n"
        f"🏛 Significance: {a['Significance']}\n"
        f"📷 DSLR Allowed: {a['DSLR Allowed']}\n"
        f"✈ Airport within 50 km: "
        f"{a['Airport with 50km Radius']}\n"
        f"👥 Google Reviews: "
        f"{a['Number of google review in lakhs']} lakhs\n"
        f"📅 Weekly Off: {a['Weekly Off']}"
    )


#SHOW DETAILS
def show_details(widget,attraction,canvas=None,content=None,details_card=None):
    if attraction is None:
        return
    widget.delete("1.0", tk.END)
    widget.insert(
        tk.END,
        details_text(attraction)
    )

    if canvas and content and details_card:
        canvas.after(50,lambda: scroll_to(canvas,details_card,content))

In [ ]:
#main window
root = tk.Tk()
root.title("Smart Tourist Attraction Explorer")
root.geometry("1150x780")
root.minsize(950, 650)
root.configure(bg=BG)

#navigation
nav = tk.Frame(root, bg=DARK, height=55)
nav.pack(fill="x")
nav.pack_propagate(False)

tk.Label(nav,text="🌍  Smart Tourist Explorer",font=("Arial", 13, "bold"),bg=DARK, fg="white").pack(side="left", padx=25)

nav_buttons = tk.Frame(nav, bg=DARK)
nav_buttons.pack(side="right", padx=20)
pages = tk.Frame(root, bg=BG)
pages.pack(fill="both", expand=True)

home = tk.Frame(pages, bg=BG)
discover = tk.Frame(pages, bg=BG)

home.place(relwidth=1, relheight=1)
discover.place(relwidth=1, relheight=1)



In [ ]:
# HOME PAGE
home_canvas, home_content = scroll_page(home)
hero = tk.Frame(home_content, bg=DARK, height=180)
hero.pack(fill="x")
hero.pack_propagate(False)

tk.Label(hero, text="SMART TRAVEL EXPERIENCE",font=("Arial", 10, "bold"),bg=DARK, fg="#A8D5CB").pack(pady=(25, 4))

tk.Label(hero, text="Discover Incredible Places",font=("Arial", 28, "bold"),bg=DARK, fg="white").pack()

tk.Label(
    hero,
    text="Explore • Compare • Discover • Plan your next destination",
    font=("Arial", 11),
    bg=DARK, fg="#D7E8E4"
).pack(pady=7)


home_body = tk.Frame(home_content, bg=BG)
home_body.pack(fill="both", padx=35, pady=25)


#WELCOME
welcome = tk.Frame(home_body, bg=LIGHT_GREEN,highlightbackground=BORDER,highlightthickness=1)
welcome.pack(fill="x", pady=(0, 20))

tk.Label(welcome,text="👋 Welcome, Explorer!",font=("Arial", 19, "bold"),bg=LIGHT_GREEN, fg=DARK).pack(pady=(18, 4))

tk.Label(
    welcome,
    text="Discover tourist attractions across India.\n"
         "Select or type a city, compare places, "
         "or use Discover for recommendations.",
    font=("Arial", 10),
    bg=LIGHT_GREEN, fg=MUTED,
    justify="center"
).pack(pady=(0, 18))


In [ ]:
# EXPLORE + TOP RATED
top_area = tk.Frame(home_body, bg=BG)
top_area.pack(fill="x")

#EXPLORE
explore = card(top_area)
explore.pack(side="left", fill="both",expand=True, padx=(0, 10))

tk.Label(explore,text="🔎  Explore Attractions",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT).pack(anchor="w", padx=25, pady=(20, 4))

tk.Label(explore,text="Choose or type a city and attraction type.",font=("Arial", 10),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25, pady=(0, 12))

tk.Label(explore, text="CITY",font=("Arial", 9, "bold"),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25)

city_combo = ttk.Combobox(explore, values=cities,state="normal", font=("Arial", 11))
city_combo.pack(fill="x", padx=25, pady=(3, 12))

tk.Label(explore, text="ATTRACTION TYPE",font=("Arial", 9, "bold"),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25)

type_combo = ttk.Combobox(explore, values=["All"],state="readonly", font=("Arial", 11))
type_combo.set("All")
type_combo.pack(fill="x", padx=25, pady=(3, 15))

#TOP RATED
top_card = card(top_area)
top_card.pack(side="right", fill="both",expand=True, padx=(10, 0))

tk.Label(top_card,text="⭐  Top Rated Attractions",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT
).pack(anchor="w", padx=25, pady=(20, 4))

tk.Label(top_card,text="Click an attraction to see its details.",font=("Arial", 10),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25, pady=(0, 10))

top_frame = tk.Frame(top_card, bg=CARD)
top_frame.pack(fill="both", expand=True,padx=20, pady=(0, 15))



In [ ]:
# HOME RESULTS
results_card = card(home_body)
results_card.pack(fill="x", pady=20)

tk.Label(results_card,text="🏛  Tourist Attractions",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT
).pack(anchor="w", padx=25, pady=(20, 4))

result_info = tk.Label(results_card,text="Search results will appear here.",font=("Arial", 10),bg=CARD, fg=MUTED)
result_info.pack(anchor="w", padx=25, pady=(0, 12))

result_list = tk.Listbox(results_card,height=8,font=("Arial", 11),bg="#FAFBFC",fg=TEXT,selectbackground=GREEN,selectforeground="white",
    relief="flat"
)
result_list.pack(fill="x", padx=25, pady=(0, 22))


#HOME DETAILS
details_card = card(home_body)
details_card.pack(fill="x", pady=(0, 20))

tk.Label(details_card,text="📍  Attraction Details",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT
).pack(anchor="w", padx=25, pady=(20, 4))

tk.Label(details_card,text="Select an attraction from the list or Top Rated section.",font=("Arial", 10),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25, pady=(0, 12))

home_details = tk.Text(details_card,height=10,font=("Arial", 10),bg="#FAFBFC",fg=TEXT,relief="flat",padx=15, pady=12)
home_details.pack(fill="x", padx=25, pady=(0, 22))


In [ ]:
# HOME FUNCTIONS
def update_types(event=None):
    city = city_combo.get().strip()
    city_data = data[
        data["City"].astype(str).str.lower()
        == city.lower()
    ]

    available = sorted(city_data["Type"].dropna().astype(str).unique())

    type_combo["values"] = ["All"] + available
    type_combo.set("All")
    update_top_rated()


def search_attractions():
    city = city_combo.get().strip()
    selected_type = type_combo.get()

    if not city:
        messagebox.showwarning("Input Required","Please select or type a city.")
        return

    result = data[
        data["City"].astype(str).str.lower()
        == city.lower()
    ]

    if result.empty:
        messagebox.showwarning("City Not Found","This city is not available in the dataset.")
        return

    if selected_type != "All":
        result = result[
            result["Type"].astype(str).str.lower()
            == selected_type.lower()
        ]

    result_list.delete(0, tk.END)
    home_details.delete("1.0", tk.END)

    if result.empty:
        result_info.config(text="No attractions found.")
        return

    # Store actual dataframe rows
    home_results.clear()
    home_results.extend(result.index.tolist())

    for _, a in result.iterrows():
        result_list.insert(tk.END, a["Name"])

    result_info.config(
        text=f"{len(result)} attraction(s) found in {city}."
    )

    home_canvas.after(50,lambda: scroll_to(home_canvas,results_card,home_content))

def update_top_rated():
    city = city_combo.get().strip()
    selected_type = type_combo.get()

    result = data.copy()

    if city:
        result = result[
            result["City"].astype(str).str.lower()
            == city.lower()
        ]

    if selected_type != "All":
        result = result[
            result["Type"].astype(str).str.lower()
            == selected_type.lower()
        ]

    result = (
        result.dropna(subset=["Google review rating"]).sort_values("Google review rating",ascending=False).head(5))
    
    clear(top_frame)

    for i, (_, a) in enumerate(result.iterrows(), 1):

        row = tk.Frame(top_frame,bg=LIGHT_GREEN,cursor="hand2")
        row.pack(fill="x", pady=3)

        tk.Label(row,text=f"{i:02d}",width=4,font=("Arial", 10, "bold"),bg=LIGHT_GREEN,fg=GREEN).pack(side="left", padx=5, pady=8)

        tk.Label(row,text=str(a["Name"]),font=("Arial", 10, "bold"),bg=LIGHT_GREEN,fg=TEXT).pack(side="left", fill="x", expand=True)

        tk.Label(row,text=f"⭐ {a['Google review rating']:.1f}",font=("Arial", 10, "bold"),bg=LIGHT_GREEN,fg="#B87900"
        ).pack(side="right", padx=8)

        # Pass the actual row, not just its name
        for w in row.winfo_children():
            w.bind(
                "<Button-1>",
                lambda e, attraction=a:
                show_details(home_details,attraction,home_canvas,home_content,details_card))
home_results = []



In [ ]:
# HOME EVENTS

city_combo.bind("<<ComboboxSelected>>", update_types)
city_combo.bind("<Return>", update_types)

type_combo.bind("<<ComboboxSelected>>",lambda e: update_top_rated())

def select_home_result(event=None):
    selected = result_list.curselection()
    if not selected:
        return

    index = home_results[selected[0]]
    attraction = data.loc[index]

    show_details(home_details,attraction,home_canvas,home_content,details_card)


result_list.bind("<<ListboxSelect>>",select_home_result)

button(explore,"🔍  Search Attractions",search_attractions).pack(pady=(0, 22))


In [ ]:
# COMPARE
compare_card = card(home_body)
compare_card.pack(fill="x", pady=(0, 20))

tk.Label(compare_card,text="⚖  Compare Attractions",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT
).pack(anchor="w", padx=25, pady=(20, 4))

tk.Label(compare_card,text="Compare two attractions using dataset information.",font=("Arial", 10),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25, pady=(0, 12))

compare_frame = tk.Frame(compare_card, bg=CARD)
compare_frame.pack(fill="x", padx=25)

compare1 = ttk.Combobox(compare_frame,values=attractions,state="readonly")
compare1.pack(side="left", fill="x",expand=True, padx=(0, 8))

compare2 = ttk.Combobox(compare_frame,values=attractions,state="readonly")
compare2.pack(side="left", fill="x",expand=True, padx=(8, 0))

def compare_attractions():
    name1 = compare1.get()
    name2 = compare2.get()

    if not name1 or not name2:
        messagebox.showwarning("Selection Required","Please select two attractions.")
        return

    if name1 == name2:
        messagebox.showwarning("Invalid Selection","Please select two different attractions.")
        return

    a1 = data[data["Name"] == name1].iloc[0]
    a2 = data[data["Name"] == name2].iloc[0]

    win = tk.Toplevel(root)
    win.title("Attraction Comparison")
    win.geometry("900x620")
    win.configure(bg=BG)

    tk.Label(win,text="⚖  Attraction Comparison",font=("Arial", 22, "bold"),bg=DARK, fg="white",pady=18
    ).pack(fill="x")

    text = tk.Text(win,font=("Courier New", 10),bg=CARD, fg=TEXT,padx=20, pady=20)
    text.pack(fill="both", expand=True, padx=30, pady=30)

    features = [
        ("City", "City"),
        ("Type", "Type"),
        ("Google Rating", "Google review rating"),
        ("Entrance Fee (INR)", "Entrance Fee in INR"),
        ("Time Needed (hours)", "time needed to visit in hrs"),
        ("Best Time to Visit", "Best Time to visit"),
        ("Significance", "Significance"),
        ("DSLR Allowed", "DSLR Allowed"),
        ("Airport within 50 km", "Airport with 50km Radius"),
        ("Google Reviews (lakhs)",
         "Number of google review in lakhs"),
        ("Weekly Off", "Weekly Off")
    ]

    text.insert(tk.END,f"{'FEATURE':<25}"f"{name1:<30}"f"{name2:<30}\n")
    text.insert(tk.END, "-" * 85 + "\n")

    for label, column in features:
        text.insert(tk.END,f"{label:<25}"f"{str(a1[column]):<30}"f"{str(a2[column]):<30}\n")
    text.config(state="disabled")

button(compare_card,"⚖  Compare Attractions",compare_attractions).pack(pady=20)

In [ ]:
# ANALYTICS
def view_analytics():
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
    win = tk.Toplevel(root)
    win.title("Tourism Analytics")
    win.geometry("1050x800")
    win.minsize(900, 650)
    win.configure(bg=BG)
    canvas, content = scroll_page(win)

    tk.Label(content,text="📊  Tourism Analytics Dashboard",font=("Arial", 24, "bold"),bg=DARK, fg="white",pady=22).pack(fill="x")

    # Statistics
    stats_frame = tk.Frame(content, bg=BG)
    stats_frame.pack(fill="x", padx=30, pady=25)

    stats = [
        ("🏛", "Total Attractions", len(data)),
        ("📍", "Total Cities", data["City"].nunique()),
        ("🏷", "Attraction Types", data["Type"].nunique())
    ]

    for icon, label, value in stats:

        c = card(stats_frame)
        c.pack(side="left",fill="both",expand=True,padx=6)

        tk.Label(c, text=icon,font=("Arial", 20),bg=CARD).pack(pady=(14, 2))

        tk.Label(c, text=str(value),font=("Arial", 20, "bold"),bg=CARD, fg=GREEN).pack()

        tk.Label(c, text=label,font=("Arial", 10),bg=CARD, fg=MUTED).pack(pady=(2, 14))


    #CHART 1
    city_counts = data["City"].value_counts().head(15)

    fig = plt.Figure(figsize=(9, 5.5), dpi=100)
    ax = fig.add_subplot(111)

    city_counts.plot(kind="bar", ax=ax)

    ax.set_title(
        "Top 15 Cities by Number of Tourist Attractions"
    )
    ax.set_xlabel("City")
    ax.set_ylabel("Number of Attractions")
    ax.tick_params(axis="x", rotation=45)

    fig.tight_layout()

    chart = FigureCanvasTkAgg(fig, master=content)
    chart.draw()
    chart.get_tk_widget().pack(pady=15)


    # CHART 2 
    type_counts = data["Type"].value_counts().head(10)
    fig = plt.Figure(figsize=(8, 6), dpi=100)
    ax = fig.add_subplot(111)
    ax.pie(type_counts.values,labels=type_counts.index,autopct="%1.1f%%",startangle=90)
    ax.set_title("Distribution of Top 10 Tourist Attraction Types")
    fig.tight_layout()
    chart = FigureCanvasTkAgg(fig, master=content)
    chart.draw()
    chart.get_tk_widget().pack(pady=15)

    # CHART 3 

    average_fee = (data.groupby("Type")["Entrance Fee in INR"].mean().sort_values(ascending=False).head(15))
    fig = plt.Figure(figsize=(9, 5.5), dpi=100)
    ax = fig.add_subplot(111)

    average_fee.plot(kind="bar", ax=ax)

    ax.set_title("Average Entrance Fee by Tourist Attraction Type")
    ax.set_xlabel("Attraction Type")
    ax.set_ylabel("Average Entrance Fee (INR)")
    ax.tick_params(axis="x", rotation=45)

    fig.tight_layout()

    chart = FigureCanvasTkAgg(fig, master=content)
    chart.draw()
    chart.get_tk_widget().pack(pady=15)

    button(content,"Close",win.destroy).pack(pady=25)


# DISCOVER PAGE

discover_canvas, discover_content = scroll_page(discover)

discover_hero = tk.Frame(discover_content,bg=DARK,height=170)
discover_hero.pack(fill="x")
discover_hero.pack_propagate(False)

tk.Label(discover_hero,text="🔎  Discover Your Next Destination",font=("Arial", 26, "bold"),bg=DARK, fg="white"
).pack(pady=(35, 5))

tk.Label(discover_hero,text="Find attractions across India based on what matters to you.",font=("Arial", 11),bg=DARK, fg="#D7E8E4"
).pack()

discover_body = tk.Frame(discover_content, bg=BG)
discover_body.pack(fill="both", padx=35, pady=25)


# DISCOVER FILTERS
filter_card = card(discover_body)
filter_card.pack(fill="x", pady=(0, 20))

tk.Label(filter_card,text="🎯  Tell us what you're looking for",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT
).pack(anchor="w", padx=25, pady=(20, 4))

tk.Label(filter_card,text="These filters search the entire dataset, not just one city.",font=("Arial", 10),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25, pady=(0, 15))


filter_grid = tk.Frame(filter_card, bg=CARD)
filter_grid.pack(fill="x", padx=20)


def filter_label(text, row, col):
    tk.Label(filter_grid,text=text,font=("Arial", 9, "bold"),bg=CARD, fg=MUTED
    ).grid(row=row, column=col,sticky="w", padx=8, pady=4)

# Rating
filter_label("Minimum Rating", 0, 0)

rating_combo = ttk.Combobox(filter_grid,values=["Any", "3.0", "3.5", "4.0", "4.2", "4.5", "4.7"],state="readonly")
rating_combo.set("Any")
rating_combo.grid(row=1, column=0,sticky="ew", padx=8, pady=(0, 12))

# Fee
filter_label("Maximum Entrance Fee (₹)", 0, 1)
fee_entry = tk.Entry(filter_grid,font=("Arial", 10),relief="flat")
fee_entry.grid(row=1, column=1,sticky="ew", padx=8, pady=(0, 12))

# Time
filter_label("Maximum Visit Time (hours)", 0, 2)

time_entry = tk.Entry(filter_grid,font=("Arial", 10),relief="flat")
time_entry.grid(row=1, column=2,sticky="ew", padx=8, pady=(0, 12))

# Type
filter_label("Attraction Type", 2, 0)

discover_type = ttk.Combobox(filter_grid,values=["All"] + types,state="readonly")
discover_type.set("All")
discover_type.grid(row=3, column=0,sticky="ew", padx=8, pady=(0, 12))

# Sorting
filter_label("Sort Results By", 2, 1)

sort_combo = ttk.Combobox(
    filter_grid,
    values=["Highest Rating","Lowest Entrance Fee","Shortest Visit Time","Most Reviewed"],
    state="readonly"
)
sort_combo.set("Highest Rating")
sort_combo.grid(row=3, column=1,sticky="ew", padx=8, pady=(0, 12))

# Checkboxes
option_frame = tk.Frame(filter_grid, bg=CARD)
option_frame.grid(row=3, column=2,sticky="w", padx=8)

dslr_var = tk.BooleanVar()
airport_var = tk.BooleanVar()

tk.Checkbutton(option_frame,text="📷 DSLR Allowed",variable=dslr_var,bg=CARD, activebackground=CARD
).pack(anchor="w")

tk.Checkbutton(option_frame,text="✈ Airport within 50 km",variable=airport_var,bg=CARD, activebackground=CARD).pack(anchor="w")

for col in range(3):
    filter_grid.columnconfigure(col, weight=1)


# DISCOVER RESULT:
discover_results = card(discover_body)
discover_results.pack(fill="x", pady=(0, 20))

tk.Label(discover_results,text="✨  Recommended Attractions",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT
).pack(anchor="w", padx=25, pady=(20, 4))

discover_info = tk.Label(discover_results,text="Set your preferences and find matching attractions.",font=("Arial", 10),bg=CARD, fg=MUTED)
discover_info.pack(anchor="w", padx=25, pady=(0, 12))

recommend_frame = tk.Frame(discover_results, bg=CARD)
recommend_frame.pack(fill="x", padx=20, pady=(0, 20))

# DISCOVER DETAILS
discover_details_card = card(discover_body)
discover_details_card.pack(fill="x", pady=(0, 20))

tk.Label(discover_details_card,text="📍  Attraction Details",font=("Arial", 17, "bold"),bg=CARD, fg=TEXT).pack(anchor="w", padx=25, pady=(20, 4))

tk.Label(discover_details_card,text="Click View Details on a recommendation.",font=("Arial", 10),bg=CARD, fg=MUTED
).pack(anchor="w", padx=25, pady=(0, 12))

discover_details = tk.Text(discover_details_card,height=10,font=("Arial", 10),bg="#FAFBFC",fg=TEXT,relief="flat",padx=15, pady=12)
discover_details.pack(fill="x", padx=25, pady=(0, 22))


# DISCOVER RECOMMENDATION
def recommendation(a):
    row = tk.Frame(recommend_frame,bg="#FAFBFC",highlightbackground=BORDER,highlightthickness=1)
    row.pack(fill="x", pady=5)

    left = tk.Frame(row, bg="#FAFBFC")
    left.pack(side="left",fill="both",expand=True,padx=15, pady=12)

    tk.Label(left,text=str(a["Name"]),font=("Arial", 12, "bold"),bg="#FAFBFC", fg=TEXT).pack(anchor="w")

    tk.Label(left,text=f"📍 {a['City']}  •  {a['Type']}",font=("Arial", 9),bg="#FAFBFC", fg=MUTED).pack(anchor="w", pady=3)

    tk.Label(left,text=f"⭐ {a['Google review rating']}    "
             f"💰 ₹{a['Entrance Fee in INR']}",
        font=("Arial", 9, "bold"),
        bg="#FAFBFC", fg=GREEN
    ).pack(anchor="w")

    button(row,"View Details →",
        lambda attraction=a:
        show_details(discover_details,attraction,discover_canvas,discover_content,discover_details_card)
    ).pack(
        side="right", padx=15
    )


# DISCOVER FILTER FUNCTION

def apply_filters():

    result = data.copy()

    # Rating
    rating = rating_combo.get()  #gets whatever user selects

    if rating != "Any":
        result = result[
            result["Google review rating"]
            >= float(rating)
        ]

    # Fee
    fee = fee_entry.get().strip()

    if fee:
        try:
            fee = float(fee)
        except ValueError:
            messagebox.showwarning(
                "Invalid Fee",
                "Please enter a valid maximum fee."
            )
            return

        result = result[
            result["Entrance Fee in INR"].notna()
            & (result["Entrance Fee in INR"] <= fee)
        ]

    # Visit time
    visit_time = time_entry.get().strip()

    if visit_time:
        try:
            visit_time = float(visit_time)
        except ValueError:
            messagebox.showwarning(
                "Invalid Time",
                "Please enter a valid visit duration."
            )
            return

        result = result[
            result["time needed to visit in hrs"].notna()
            & (
                result["time needed to visit in hrs"]
                <= visit_time
            )
        ]

    # Attraction type
    selected_type = discover_type.get()

    if selected_type != "All":
        result = result[
            result["Type"].astype(str).str.lower()
            == selected_type.lower()
        ]

    # DSLR
    if dslr_var.get():
        result = result[
            result["DSLR Allowed"]
            .astype(str).str.lower()
            .isin(["yes", "true"])
        ]

    # Airport
    if airport_var.get():
        result = result[
            result["Airport with 50km Radius"]
            .astype(str).str.lower()
            .isin(["yes", "true"])
        ]

    # Sorting
    sort = sort_combo.get()

    if sort == "Highest Rating":
        result = result.sort_values(
            "Google review rating",
            ascending=False
        )

    elif sort == "Lowest Entrance Fee":
        result = result.sort_values(
            "Entrance Fee in INR"
        )

    elif sort == "Shortest Visit Time":
        result = result.sort_values(
            "time needed to visit in hrs"
        )

    elif sort == "Most Reviewed":
        result = result.sort_values(
            "Number of google review in lakhs",
            ascending=False
        )

    
    clear(recommend_frame)
    discover_details.delete("1.0", tk.END)

    if result.empty:

        discover_info.config(
            text="No attractions match your preferences."
        )

        tk.Label(recommend_frame,text="😔 No matching attractions found.",font=("Arial", 12, "bold"),bg=CARD, fg=MUTED).pack(pady=30)
        return

    total = len(result)
    result = result.head(15)

    discover_info.config(
        text=f"{total} attraction(s) found. "
             f"Showing top {len(result)}."
    )

    for _, a in result.iterrows():
        recommendation(a)

    # Automatically show results
    discover_canvas.after(50,lambda: scroll_to(discover_canvas,discover_results,discover_content))


# RESET DISCOVER

def reset_discover():
    rating_combo.set("Any")
    fee_entry.delete(0, tk.END)
    time_entry.delete(0, tk.END)

    discover_type.set("All")
    sort_combo.set("Highest Rating")

    dslr_var.set(False)
    airport_var.set(False)

    clear(recommend_frame)
    discover_details.delete("1.0", tk.END)

    discover_info.config(
        text="Set your preferences and find matching attractions."
    )

    discover_canvas.yview_moveto(0)


button(filter_card,"✨  Find Attractions",apply_filters).pack(pady=(0, 10))

tk.Button(filter_card,text="Clear Filters",command=reset_discover,font=("Arial", 9, "bold"),bg="white",fg=TEXT,relief="flat",
    cursor="hand2",
    padx=18, pady=6
).pack(pady=(0, 20))


In [ ]:
# NAVIGATION FUNCTIONS

def show_home():
    home.lift()
    home_canvas.yview_moveto(0)

def reset_home():
    city_combo.set("")
    type_combo["values"] = ["All"]
    type_combo.set("All")

    result_list.delete(0, tk.END)
    home_details.delete("1.0", tk.END)

    result_info.config(
        text="Search results will appear here."
    )

    update_top_rated()
    show_home()


#it is responsible for opening discover page
def show_discover(): 
    discover.lift()  #bring the discover frame to front
    discover_canvas.yview_moveto(0)  #move discover page scrollbar to top




In [ ]:
# NAVIGATION BUTTONS

tk.Button(nav_buttons,text="🏠 Home",command=reset_home,font=("Arial", 9, "bold"),bg=DARK, fg="white",activebackground=GREEN_DARK,
    relief="flat", cursor="hand2",
    padx=10
).pack(side="left", padx=3)

tk.Button(nav_buttons,text="🔎 Discover",command=show_discover,font=("Arial", 9, "bold"),bg=DARK, fg="white",activebackground=GREEN_DARK,
    relief="flat", cursor="hand2",
    padx=10
).pack(side="left", padx=3)

tk.Button(nav_buttons,text="📊 Analytics",command=view_analytics,font=("Arial", 9, "bold"),bg=DARK, fg="white",activebackground=GREEN_DARK,
    relief="flat", cursor="hand2",
    padx=10
).pack(side="left", padx=3)

In [ ]:
# FOOTERS
tk.Label(home_content,text="Smart Tourist Attraction Explorer  •  ""Data-driven tourism exploration",font=("Arial", 9),bg=DARK, fg="#D7E8E4",pady=12
).pack(fill="x")

tk.Label(discover_content,text="Smart Tourist Attraction Explorer  •  ""Data-driven recommendations",font=("Arial", 9),bg=DARK, fg="#D7E8E4",pady=12
).pack(fill="x")



# START
update_top_rated()
show_home()

root.mainloop()